##Bronze Layer (Raw Ingestion)

In [0]:
# Read csv
df=spark.read.format('csv')\
             .option('header','True')\
             .option('inferschema','true')\
             .load('/Volumes/workspace/default/csv/PS_20174392719_1491204439457_log.csv')
df.display()

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
1,TRANSFER,181.0,C1305486145,181.0,0.0,C553264065,0.0,0.0,1,0
1,CASH_OUT,181.0,C840083671,181.0,0.0,C38997010,21182.0,0.0,1,0
1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0
1,PAYMENT,7817.71,C90045638,53860.0,46042.29,M573487274,0.0,0.0,0,0
1,PAYMENT,7107.77,C154988899,183195.0,176087.23,M408069119,0.0,0.0,0,0
1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.0,0,0
1,PAYMENT,4024.36,C1265012928,2671.0,0.0,M1176932104,0.0,0.0,0,0
1,DEBIT,5337.77,C712410124,41720.0,36382.23,C195600860,41898.0,40348.79,0,0


In [0]:
df.describe()
df.printSchema()
print('Rows:',df.count(),'Columns:',len(df.columns))

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)

Rows: 6362620 Columns: 11


##Write Bronze Table

In [0]:
from pyspark.sql.functions import current_timestamp

df_bronze = df.withColumn("ingestion_time", current_timestamp())

In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_transaction")


In [0]:
#Verify delta table
spark.sql("DESCRIBE DETAIL bronze_transaction").show()

+------+--------------------+--------------------+-----------+--------+--------------------+-------------------+----------------+-----------------+--------+-----------+--------------------+----------------+----------------+--------------------+--------------------+-------------+
|format|                  id|                name|description|location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|          properties|minReaderVersion|minWriterVersion|       tableFeatures|          statistics|clusterByAuto|
+------+--------------------+--------------------+-----------+--------+--------------------+-------------------+----------------+-----------------+--------+-----------+--------------------+----------------+----------------+--------------------+--------------------+-------------+
| delta|6efab855-8ec7-474...|workspace.default...|       NULL|        |2026-03-14 17:02:...|2026-03-16 08:05:40|              []|               []|       8|  19

In [0]:
%sql
select count(*)  from bronze_transaction

count(*)
6362620


In [0]:
df_bronze = spark.table('workspace.default.bronze_transaction').repartition(8)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_transaction")

In [0]:
spark.sql("DESCRIBE HISTORY bronze_transaction").show()

+-------+-------------------+--------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|        userId|            userName|           operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+--------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      1|2026-03-16 08:05:40|77100717911144|shaheer2226sb@gma...|CREATE OR REPLACE...|{partitionBy -> [...|NULL|{2358057492245315}|   456c8a6c-ee1a-4de...|0316-080354-q5e6q